Author: Nitish Arora

READ ME:

1. This notebook leverages Python script to create the Uplift factor
2. The input PropTrack datasets need to be saved in this location:
- hotspotting_ppi/hotspotting_quarterly_extract.csv
- hotspotting_ppi/hotspotting_sales_forecast.csv
- An existing suburb dataset (pptr_suburb) is used here for weights. This doesn't need frequent updates

In [33]:
#importing packages and setting conditions
import pandas as pd
import numpy as np

In [34]:
#Cell 1: Load data

#Importing key python library
from google.cloud import bigquery

#Defining the client
client = bigquery.Client()

#Defining the queries for bigquery. The PropTrack table is dropped here and re-uploaded for analysis
query = """
drop table `ppi.data_qtr`;
drop table `ppi.data_qtr_uplift`;
drop table `ppi.data_qtr_fc`;
drop table `ppi.data_sb`;
-- Data load - load the latest data called 'hotspotting_quarterly_extract.csv' in the hotspotting_ppi_data bucket
CREATE EXTERNAL TABLE `ppi.data_qtr_uplift`
OPTIONS (
  FORMAT = 'CSV',
  uris = ['gs://hotspotting_ppi/hotspotting_quarterly_extract.csv']
);

-- Data load - load the latest data called 'hotspotting_sales_forecast.csv' in the hotspotting_ppi_data bucket to use for uplift
CREATE EXTERNAL TABLE `ppi.data_qtr_fc`
OPTIONS (
  FORMAT = 'CSV',
  uris = ['gs://hotspotting_ppi/hotspotting_sales_forecast.csv']
);

-- Data load - load the latest data called 'pptr_suburb.csv' in the hotspotting_ppi_data bucket to use for uplift
CREATE EXTERNAL TABLE `ppi.data_sb`
OPTIONS (
  FORMAT = 'CSV',
  uris = ['gs://hotspotting_ppi/pptr_suburb.csv']
);
"""

#Running the queries with the results
query_job = client.query(query)
results = query_job.result()

In [35]:
#Running the query to save forecast data as a python dataframe
client = bigquery.Client()

query = """
SELECT *
FROM `ppi.data_qtr_fc`
"""
query_job = client.query(query)
results = query_job.result()

#Saving the qtrly data as a python dataframe
fc = results.to_dataframe()

# Get the Forecast - Actual ratio in latest slice for each line item at prop_type level
# Note, we assume this will become equal in 3-6 months but apply right now

# #Remove dwelling
fc2 = fc[fc['property_type'] != 'DWELLING']
#Only limit the data to the latest entry
fc2 = fc2.sort_values(by=['gcc_name', 'property_type','end_date'])
# Drop duplicates, keeping the last (latest) entry
# fc3 = fc2.drop_duplicates(subset=['gcc_name', 'property_type'], keep='last')
fc2b = fc2.groupby(['gcc_name', 'property_type'])
fc3 = fc2b.nth(-2).reset_index()#To get prev qtr end data

# Make sure fc3 is the original DataFrame (not a slice)
fc3 = fc3.copy()

#Create the ratio to compare Forecast to Actuals
fc3['Uplift_Ratio'] = fc3['Sales_Volume_Forecast___3_Months___Aggregated_Value'].div(fc3['Volume_of_sales___3_Month___Calculation_Value'], fill_value=np.nan)

# fc3 = fc3.sort_values(by='Uplift_Ratio')
# fc3.tail(20)

In [36]:
#Running the query to save weights data as a python dataframe
client = bigquery.Client()

query = """
SELECT *
FROM `ppi.data_sb`
"""
query_job = client.query(query)
results = query_job.result()

#Saving the qtrly data as a python dataframe
sb = results.to_dataframe()

# Join/expand the line item to a suburb level for use in data
fc_sb = pd.merge(fc3,sb, left_on='gcc_name', right_on='gccsa_name', how='left')
fc_sb.copy()
fc_sb_n = fc_sb[['gccsa_name','sal_code', 'sal_name', 'property_type', 'Uplift_Ratio', 'end_date']].copy()

In [37]:
#Running the query to save qtrly data as a python dataframe
client = bigquery.Client()

query = """
SELECT *
FROM `ppi.data_qtr_uplift`
"""
query_job = client.query(query)
results = query_job.result()

#Saving the qtrly data as a python dataframe
df = results.to_dataframe()

In [38]:
# Applying the ratio at a suburb level from step 2 to quarterly sales data to get the results in a raw data file
# #Remove dwelling
df = df[df['property_type'] != 'DWELLING']

#Fix up the data types for the new dataset
fc_sb_n.loc[:, 'sal_code'] = fc_sb_n['sal_code'].astype('int64')
fc_sb_n['end_date'] = pd.to_datetime(fc_sb_n['end_date'])
fc_sb_n.copy()

#Merge the dataframes to get the ratio
df2 = pd.merge(df,fc_sb_n, left_on=['suburb_code','property_type'], right_on=['sal_code','property_type'], how='left')

#Apply sales ratio to the corresponding column
def manipulate_quarter_column(row):
    date_str = row['end_date'].strftime('%Y_%m_%d')  # Format the date string
    quarter_column_name = f'quarter_{date_str}'  # Create the column name dynamically

    #Get the Uplift Ratio column
    df2[f'Post_Uplift_{date_str}'] = df2[quarter_column_name] * df2['Uplift_Ratio']
    df2[f'Post_Uplift_{date_str}'] = round(df2[f'Post_Uplift_{date_str}'], 0)  # Round to 0 decimal places

df2.apply(manipulate_quarter_column, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
17284,None
17285,None
17286,None
17287,None


In [39]:
#For checks if needed
df2.head()
# fc3.tail(20)

,suburb_name,suburb_code,postcode,lga,state,property_type,metric,quarter_2021_03_31,quarter_2021_06_30,quarter_2021_09_30,...,quarter_2025_03_31,quarter_2025_06_30,quarter_2025_09_30,quarter_2025_12_31,gccsa_name,sal_code,sal_name,Uplift_Ratio,end_date,Post_Uplift_2025_12_31
0,Aberdeen (NSW),10007,2336,Upper Hunter Shire,New South Wales,HOUSE,sales_volume_3mths,18.0,11.0,9.0,...,10.0,7.0,14.0,12.0,Rest of NSW,10007,Aberdeen,1.056839,2025-12-31,13.0
1,Aberglasslyn,10009,2320,Maitland,New South Wales,HOUSE,sales_volume_3mths,57.0,44.0,50.0,...,40.0,28.0,39.0,32.0,Rest of NSW,10009,Aberglasslyn,1.056839,2025-12-31,34.0
2,Albion Park,10025,2527,Shellharbour,New South Wales,HOUSE,sales_volume_3mths,74.0,77.0,81.0,...,45.0,56.0,49.0,52.0,Rest of NSW,10025,Albion Park,1.056839,2025-12-31,55.0
3,Albion Park,10025,2527,Shellharbour,New South Wales,UNIT,sales_volume_3mths,14.0,20.0,28.0,...,16.0,26.0,13.0,24.0,Rest of NSW,10025,Albion Park,1.094937,2025-12-31,26.0
4,Alexandria,10030,2015,Sydney,New South Wales,UNIT,sales_volume_3mths,90.0,79.0,72.0,...,54.0,70.0,84.0,78.0,Greater Sydney,10030,Alexandria,1.150842,2025-12-31,90.0


In [40]:
#Export the dataset for use in other scripts
#Import the required python library
import pytz

#Selecting the columns from the final analysis

# Create a BigQuery client
client = bigquery.Client() #Ensure prev table is dropped before this is run

# Create a table with the specified schema
table_id = 'ppi.data_qtr'
job = client.load_table_from_dataframe(
    df2, table_id
)  # Make an API request.
job.result()  # Wait for the job to complete.

LoadJob<project=ppi-automation, location=asia-southeast1, id=198c42a4-9384-41d2-9cc9-3c547e2e6a3d>

Appendix and checks

In [41]:
# #Check against existing uplift.csv to compare results.
# #Running the query to save qtrly data as a python dataframe
# client = bigquery.Client()

# query = """
# SELECT *
# FROM `ppi.data_qtr`
# """
# query_job = client.query(query)
# results = query_job.result()

# #Saving the qtrly data as a python dataframe
# upl = results.to_dataframe()

# upl2 = pd.merge(df2,upl, on=['suburb_code','property_type'])

In [42]:
# #Checks
# # upl.count() - Numbers similar for df2 and upl

# #Comparing the Uplift numbers jic
# upl3 = upl2#[['suburb_name_x',	'suburb_code','property_type', 'Uplift_Ratio_x','Post_Uplift_2024_12_31_x',
#             #  'Uplift_Ratio_y','Post_Uplift_2024_12_31_y']]

# #Compare uplift
# def up_ch(df):
#   if (df['Post_Uplift_2024_12_31_x'] ==df['Post_Uplift_2024_12_31_y']):
#     return 'Yes'
#   else:
#     return 'No'

# upl3['ch'] = upl3.apply(up_ch, axis=1)

# # upl4 = upl3.groupby(['ch'])['suburb_code'].count().reset_index()
# # upl4.head()

# upl5 = upl3[upl3['ch']=='No']

# # print(upl5)

In [43]:
# Suburb data checks

#Check to make sure all entries are in the finald dataset (try with right join too for checks)
# unmatched_fc3 = fc3[~fc3['gcc_name'].isin(fc_sb['gcc_name'])]
# unmatched_fc3.head()

#Check for lengths of resulting dfs
# print(len(fc3))
# print(len(sb))
# print(len(fc_sb))

#Check to make sure all gcc of right join dataset is in fact not in Aus or in PPT data = confirmed, result is:
# ['No usual address' 'Migratory - Offshore - Shipping' 'Other Territories'
#  'Outside Australia']
#About 34 entries for these
# distinct_gccsa_for_null_gcc = fc_sb.loc[fc_sb['gcc_name'].isnull(), 'gccsa_name'].unique()
# distinct_gccsa_for_null_gcc.copy()
# print(distinct_gccsa_for_null_gcc)

# fc_sb.head()
# print(fc3.columns)  # Check columns in fc3
# print(sb.columns)   # Check columns in
# print(fc_sb.columns)  # Check columns in fc3